In [8]:
# Resume Parser for Single File (Jupyter - One Cell)

import sys
import subprocess
import importlib.util
import re
import os
from pathlib import Path

# ---------- Auto-install required packages ----------
required_packages = {
    "PyPDF2": "PyPDF2",
    "docx": "python-docx",
    "pandas": "pandas"
}

for import_name, pip_name in required_packages.items():
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])

import PyPDF2
import docx
import pandas as pd

# ---------- Single File Path ----------
file_path = r"D:\VS Code files\SEM 4\NLP & AI\Projects\resume_dataset.pdf"
# Change resume.pdf to your real file name

# ---------- Skill & Education Keywords ----------
SKILL_KEYWORDS = [
    "python", "java", "c", "c++", "c#", "javascript", "typescript",
    "html", "css", "sql", "mysql", "postgresql", "mongodb",
    "flask", "django", "fastapi", "react", "node", "nodejs", "express",
    "bootstrap", "tailwind", "pandas", "numpy", "matplotlib", "seaborn",
    "scikit-learn", "sklearn", "tensorflow", "keras", "pytorch",
    "machine learning", "deep learning", "nlp", "natural language processing",
    "data science", "data analysis", "computer vision",
    "excel", "power bi", "tableau", "git", "github", "linux", "aws",
    "azure", "docker", "kubernetes", "cybersecurity", "ethical hacking",
    "network security", "iot", "arduino", "raspberry pi"
]

EDUCATION_KEYWORDS = [
    "b.sc", "bsc", "b.e", "be", "b.tech", "btech",
    "m.sc", "msc", "m.e", "me", "m.tech", "mtech",
    "phd", "doctorate", "diploma",
    "computer science", "information technology",
    "artificial intelligence", "machine learning", "data science"
]

# ---------- Text Extraction ----------
def extract_text_from_pdf(path):
    text = ""
    try:
        with open(path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
    except Exception as e:
        print("PDF read error:", e)
    return text

def extract_text_from_docx(path):
    try:
        doc = docx.Document(path)
        return "\n".join([para.text for para in doc.paragraphs])
    except Exception as e:
        print("DOCX read error:", e)
        return ""

def extract_text(path):
    ext = Path(path).suffix.lower()
    if ext == ".pdf":
        return extract_text_from_pdf(path)
    elif ext == ".docx":
        return extract_text_from_docx(path)
    elif ext == ".txt":
        try:
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                return f.read()
        except Exception as e:
            print("TXT read error:", e)
            return ""
    else:
        raise ValueError("Unsupported file format. Use PDF, DOCX, or TXT.")

# ---------- Extraction Functions ----------
def extract_email(text):
    match = re.search(r'[\w\.-]+@[\w\.-]+\.\w+', text)
    return match.group(0) if match else "Not found"

def extract_phone(text):
    match = re.search(r'(\+?\d{1,3}[-.\s]?)?(\(?\d{3,5}\)?[-.\s]?)?\d{3,5}[-.\s]?\d{4,6}', text)
    return match.group(0) if match else "Not found"

def extract_links(text):
    links = re.findall(
        r'(https?://[^\s]+|www\.[^\s]+|linkedin\.com/[^\s]+|github\.com/[^\s]+)',
        text,
        re.IGNORECASE
    )
    unique_links = []
    for link in links:
        if link not in unique_links:
            unique_links.append(link)
    return unique_links if unique_links else ["Not found"]

def extract_name(text):
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    for line in lines[:8]:
        if len(line.split()) in [2, 3]:
            if not any(char.isdigit() for char in line):
                if not re.search(r'@|resume|cv|curriculum|vitae|phone|email|github|linkedin', line, re.IGNORECASE):
                    return line.title()
    return "Not found"

def extract_skills(text):
    text_lower = text.lower()
    found_skills = []
    for skill in SKILL_KEYWORDS:
        pattern = r'\b' + re.escape(skill.lower()) + r'\b'
        if re.search(pattern, text_lower):
            found_skills.append(skill)
    return sorted(set(found_skills), key=str.lower) if found_skills else ["Not found"]

def extract_education(text):
    text_lower = text.lower()
    found_education = []
    for edu in EDUCATION_KEYWORDS:
        if edu in text_lower:
            found_education.append(edu)
    return sorted(set(found_education), key=str.lower) if found_education else ["Not found"]

def extract_experience(text):
    text_lower = text.lower()

    years = re.findall(r'(\d+(?:\.\d+)?)\s*\+?\s*(?:years|year|yrs|yr)', text_lower)
    if years:
        years_float = [float(y) for y in years]
        return f"{max(years_float)} years"

    experience_words = ["intern", "internship", "experience", "worked", "developer", "engineer", "project"]
    count = sum(text_lower.count(word) for word in experience_words)

    if count >= 8:
        return "Likely experienced"
    elif count >= 3:
        return "Some experience"
    else:
        return "Fresher / Not clearly mentioned"

# ---------- Main ----------
if not os.path.exists(file_path):
    print("❌ File not found")
    print("Check this path:")
    print(file_path)
else:
    raw_text = extract_text(file_path)

    if not raw_text.strip():
        print("❌ No readable text found in file")
        print("This can happen if the PDF is scanned as an image.")
    else:
        result = {
            "File Name": os.path.basename(file_path),
            "Name": extract_name(raw_text),
            "Email": extract_email(raw_text),
            "Phone": extract_phone(raw_text),
            "Skills": ", ".join(extract_skills(raw_text)),
            "Education": ", ".join(extract_education(raw_text)),
            "Experience": extract_experience(raw_text),
            "Links": ", ".join(extract_links(raw_text)),
            "Text Preview": raw_text[:1000] + ("..." if len(raw_text) > 1000 else "")
        }

        df = pd.DataFrame([result])

        print("✅ Resume Parsed Successfully\n")
        for key, value in result.items():
            if key != "Text Preview":
                print(f"{key}: {value}")

        print("\n--- Parsed Data Table ---")
        display(df)

✅ Resume Parsed Successfully

File Name: resume_dataset.pdf
Name: Marketing Manager
Email: hello@reallygreatsite.com
Phone: +123-456-7890
Skills: Not found
Education: me
Experience: Fresher / Not clearly mentioned
Links: www.reallygreatsite.com

--- Parsed Data Table ---


,File Name,Name,Email,Phone,Skills,Education,Experience,Links,Text Preview
0,resume_dataset.pdf,Marketing Manager,hello@reallygreatsite.com,+123-456-7890,Not found,me,Fresher / Not clearly mentioned,www.reallygreatsite.com,EDUCATIONRICHARDSANCHEZ\nMARKETING MANAGER \nC...
